In [1]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [21]:
import tomllib

import numpy as np
import matplotlib.pyplot as plt

from src.sim_utils import generate_condition_space
from src.plot_utils import convert_size
from src.stimulus_generator import StimulusGenerator


In [3]:
def load_configurations():
    """
    Load the model, stimulus, simulation, and experiment parameters.

    Returns
    -------
    stimulus_parameters : dict
        The stimulus parameters.
    experiment_parameters : dict
        The experiment parameters.
    """
    parameters = {}

    with open('../config/simulation/stimulus.toml', 'rb') as f:
        parameters['stimulus'] = tomllib.load(f)

    with open('../config/analysis/experiment_actual.toml', 'rb') as f:
        parameters['experiment'] = tomllib.load(f)

    return parameters['stimulus'], parameters['experiment']


In [42]:
stimulus_parameters, experiment_parameters = load_configurations()
stimulus_parameters['annulus_resolution'] = stimulus_parameters['annulus_resolution'] * 10
stimulus_parameters['stimulus_resolution'] = stimulus_parameters['stimulus_resolution'] * 10
stimulus_parameters['stimulus_num_pixels'] = stimulus_parameters['stimulus_resolution'] ** 2
condition_space = generate_condition_space(experiment_parameters)

generator = StimulusGenerator(stimulus_parameters)

In [39]:
figure_size = (90, 90)  # mm
figure_size = convert_size(*figure_size)
save_path = '../results/figures/figure01_conditions/'
os.makedirs(save_path, exist_ok=True)

In [ ]:
for grid_coarseness, contrast_heterogeneity in zip(*condition_space):
    stimulus = generator.generate(
        grid_coarseness,
        contrast_heterogeneity,
        0.7
    )

    plt.figure(figsize=figure_size)
    plt.imshow(stimulus, cmap='gray', vmin=0, vmax=1)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            save_path,
            f'figure01_condition_gc{grid_coarseness}_ch{contrast_heterogeneity}.svg'
        ),
        format='svg',
    )
    plt.close()

    